# 交割单与模式样本账本：核对记录

## tl;dr

截至2026-10-23，查到495笔买卖成交，覆盖95个证券代码。10/23为9笔；10/22的7笔补齐持仓连接。短期旧档完全重叠，不重复追加。真实成交、事后补记、程序技术标签分别保存。

此文件是数据核对伴随材料，不评估策略有效性，不作交易建议。

## Context & Methods

基准是7月—10月旧XLSX。10/19—23使用2026-10-26从已登录东方财富客户端读取的交割单与历史成交两页各33行；只读原文件，未修改生产持仓和资金台账。

### Key Assumptions

用证券代码识别证券；保留历史名称。证券买卖与新股入账、资金划转分开计数。相同成交在交割页和历史成交页仅计一次；发生时间与成交时间分别保留。按库存清零划轮，不按股票名字永久绑定一个模式。历史模式缺失不以MA5等技术特征替代。

运行要求：Python 3、openpyxl。此笔记本的代码已在独立Python命名空间从上到下执行并保存输出；当前环境无nbformat/nbclient，未进行Jupyter内核兼容性验证。

## Data
### 1. 读取原始来源与计算哈希

In [ ]:
from pathlib import Path
from collections import Counter, defaultdict
from decimal import Decimal
import hashlib, json
import openpyxl

base = Path('/Users/shibicheng/shilun_standalone')
archive = base / 'outputs/01a14866-b9df-7663-a58b-dccf6cb44cd5'
old_path = archive / '东财交割单_20260701-20261017.xlsx'
short_path = archive / '东财交割单_20261010-20261017.xlsx'
excerpt_path = base / 'outputs/01a17178-dec2-7a91-91e1-cc7e68aa661e/券商查询摘录_20261019-20261023.json'

def read_statement(path):
    w = openpyxl.load_workbook(path, data_only=True, read_only=True)
    s = w['交割单']
    headers = [s.cell(5, c).value for c in range(1, 18)]
    rows = [dict(zip(headers, row)) for row in s.iter_rows(min_row=6, max_col=17, values_only=True)
            if row[0] and row[5]]
    w.close()
    return rows

old = read_statement(old_path)
short = read_statement(short_path)
pages = json.loads(excerpt_path.read_text())['pages']
statements = [dict(zip(p['headers'], row)) for k, p in pages.items() if 'settlement' in k for row in p['rows']]
deals = [dict(zip(p['headers'], row)) for k, p in pages.items() if 'deals' in k for row in p['rows']]
for p in (old_path, short_path, excerpt_path):
    print(p.name, hashlib.sha256(p.read_bytes()).hexdigest())
print('旧档财务事件', len(old), '短档财务事件', len(short), '新读交割单', len(statements), '历史成交', len(deals))
assert len(old) == 473 and len(short) == 18 and len(statements) == len(deals) == 33


## Results
### 2. 核对重叠、成交身份及交割金额

In [ ]:
def canonical(v):
    if isinstance(v, (int, float)):
        return str(Decimal(str(v)).normalize())
    return str(v or '')

old_counter = Counter(tuple(canonical(v) for v in r.values()) for r in old)
short_counter = Counter(tuple(canonical(v) for v in r.values()) for r in short)
assert not (short_counter - old_counter), '短档出现旧档没有的财务事件'
assert len({r['成交编号'] for r in deals}) == 33, '成交编号重复，需人工核对'

def money_key(r, deal=False):
    return (r['成交日期'] if deal else r['发生日期'], str(r['证券代码']).zfill(6),
            r['委托方向'] if deal else r['买卖类别'], Decimal(str(r['成交数量'])),
            Decimal(str(r['成交价格'])), Decimal(str(r['成交金额'])),
            *(Decimal(str(r[k])) for k in ['佣金', '交易规费', '印花税', '过户费']),
            Decimal(str(r['股份余额'] if deal else r['股票余额'])))

assert Counter(money_key(r) for r in statements) == Counter(money_key(r, True) for r in deals)
for r in statements:
    fee = sum(Decimal(r[k]) for k in ['佣金','交易规费','印花税','过户费'])
    expected = (Decimal(r['成交金额']) if r['买卖类别'] == '证券卖出' else -Decimal(r['成交金额'])) - fee
    assert expected == Decimal(r['发生金额'])

print('短档18条财务事件全部包含在旧档；33笔成交经济字段及交割净额均匹配。')
print('匹配不使用发生时间冒充成交时间；新页10/19—21的17笔替代事实读取，不与旧近档重复相加。')
print('每日成交笔数', dict(sorted(Counter(r['成交日期'] for r in deals).items())))
print('10月20日现已读取实际费用', sum(sum(Decimal(r[k]) for k in ['佣金','交易规费','印花税','过户费']) for r in deals if r['成交日期']=='2026-10-20'))


### 3. 全量成交与证券覆盖

In [ ]:
old_fills = [r for r in old if r['买卖类别'] in ('证券买入', '证券卖出')]
all_fills = [{'date':str(r['发生日期'])[:10], 'code':str(r['证券代码']).zfill(6), 'name':r['证券名称'], 'side':r['买卖类别']} for r in old_fills]
all_fills += [{'date':r['成交日期'], 'code':r['证券代码'], 'name':r['证券名称'], 'side':r['委托方向']} for r in deals]
securities = defaultdict(lambda: {'names':set(), 'dates':[], 'fills':0})
for r in all_fills:
    s = securities[r['code']]; s['names'].add(r['name']); s['dates'].append(r['date']); s['fills'] += 1
assert len(old_fills)==462 and len(all_fills)==495 and len(securities)==95
print('旧档买卖成交', len(old_fills), '合并后', len(all_fills), '证券代码', len(securities))
print('买卖方向', dict(Counter(r['side'] for r in all_fills)))
print('按成交笔数最多的12个证券（不代表模式完整度）')
for c, s in sorted(securities.items(), key=lambda x:(-x[1]['fills'], x[0]))[:12]:
    print(c, '/'.join(sorted(s['names'])), s['fills'], min(s['dates']), max(s['dates']))
print('名称变化按代码合并，例如 XD山东黄/山东黄金。95证券包含4只ETF，不称95只股票。')


### 4. 10月23日逐笔事实与缺口
下表只列真实成交；模式、预期、确认与退出均待用户补记。

In [ ]:
oct23 = sorted([r for r in deals if r['成交日期']=='2026-10-23'], key=lambda r:r['成交时间'])
assert len(oct23)==9
for r in oct23:
    print(r['成交时间'], r['证券名称'], r['委托方向'], r['成交数量'], r['成交价格'], '成交后库存',r['股份余额'])
wind_buys = [r for r in oct23 if r['证券代码']=='000636' and r['委托方向']=='证券买入']
qty = sum(Decimal(r['成交数量']) for r in wind_buys)
gross = sum(Decimal(r['成交金额']) for r in wind_buys)
fees = sum(sum(Decimal(r[k]) for k in ['佣金','交易规费','印花税','过户费']) for r in wind_buys)
assert len(wind_buys)==7 and qty==1700 and gross==64709 and fees==35
print('风华新轮7次买入', qty, '股，成交额', gross, '含费投入', gross+fees, '期末尚未清仓。')
print('待补：卖清依据；重新入场的独立信号；加仓是否预定/每笔确认；上限调整时间；预期窗口与失败处理。')
print('九阳待补：10/22买入模式/触发/预期，以及10/23实际卖出原因。')


### 5. 用交易事实核对清仓轮，而非合并所有同名交易

In [ ]:
def fee(r):
    return sum(Decimal(r[k]) for k in ['佣金','交易规费','印花税','过户费'])
def net(r):
    return (Decimal(r['成交金额']) if r['委托方向']=='证券卖出' else -Decimal(r['成交金额'])) - fee(r)

wind22 = [r for r in deals if r['证券代码']=='000636' and ((r['成交日期']=='2026-10-22' and r['委托方向']=='证券买入') or (r['成交日期']=='2026-10-23' and r['委托方向']=='证券卖出'))]
joyoung = [r for r in deals if r['证券代码']=='002242']
assert sum(net(r) for r in wind22)==Decimal('-3298.000')
assert sum(net(r) for r in joyoung)==Decimal('79.610')
print('风华10/22重新建仓—10/23卖清完整轮：含费现金盈亏', sum(net(r) for r in wind22))
print('九阳10/22—10/23完整轮：含费现金盈亏', sum(net(r) for r in joyoung))
print('这些是个案真实结果，不是模式期望值；风华10/23买回的新轮不混入上一轮。')


## Takeaways

1. 第一优先级：成交事实→库存轮→逐次决策补记→预期及失效条件→结果归因。按证券代码与交易轮保存，名字只作显示。
2. 国芳9/16原意是翘板；旧程序标签是日线特征，不能覆盖意图。风华不同日期也不能共用一个永久模式。
3. 金健10/19加仓延续旧轮，风华10/23清仓后重新建仓另开轮。哈药多次买卖需按实际库存而非“旧仓卖完”划轮。
4. 新华首阴、科技回流参考他人、持仓做T、MA5形态候选分别存。外部参考是决策来源；做T是管理动作；接近MA5是客观特征。
5. 已有补记直接复用；未记的字段允许未知。事后补记不能变成事前冻结计划。
6. 先用于还原自己的实际交易与检验执行；模式有效性另用完整候选全集和未来验证样本检验，不能用挑选的几个案例计算胜率。

## 本次用户补记与剩余缺口

以下内容来自本聊天事后补记。它能够还原用户意图，不能证明23日前已冻结计划。全仓是用户回忆的计划上限，不是系统建议，也不是已核算的实际仓位比例。

In [ ]:
out_dir = base / 'outputs/01a17178-dec2-7a91-91e1-cc7e68aa661e'
supplement = json.loads((out_dir / 'user_supplement_20261026.json').read_text())
wind = supplement['decisions'][1]
assert wind['cap_change_date'] == '2026-10-22'
assert wind['planned_additions'] is True
print('风华：提前计划分批买入；22日将上限调整为单一标的全仓。')
print('用户当时预期：', wind['expectation'])
print('用户低开分支原话：', wind['monday_low_open_exit_original'])
print('剩余可核验缺口：')
for item in wind['missing']: print('-', item)
print('九阳：用户原模式抱团；实际退出原因是为补风华腾资。')
print('新易盛22日：用户补记为加速下跌止损；原具体退出触发仍未知。')
print('1000亿日终规则已查到；未核实23日上午每次下单前的实时量能条件。')


### 全量库存轮索引的边界

In [ ]:
episode_audit = json.loads((out_dir / 'independent_full_episode_summary.json').read_text())
assert episode_audit['all_fills_n'] == 495
assert episode_audit['episodes_n'] == 171
assert episode_audit['episode_status_counts'] == {'closed':170, 'open':1}
assert episode_audit['inventory_errors'] == []
print('库存轮171：170轮清仓、1轮未清仓；清仓轮中1轮期初成本未知。')
print('2轮IPO与自主二级市场交易分开；红利和后续红利税不混入纯买卖现金盈亏。')
print('旧金健9/24依据券商股票余额校正库存处理顺序，原发生时间未修改。')
print('库存链可对账不等于旧档全部秒级撮合时间已核验；库存轮也不等于独立模式样本。')
